# Modeling

## Import Dependencies

In [2]:
import os
import warnings
from time import time

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import log_loss, accuracy_score, precision_score, recall_score, f1_score
import optuna
import mlflow

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

## Data Loading

In [3]:
data = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "data"))
experiments = os.path.abspath(os.path.join(os.path.dirname(__name__), "..", "experiments"))

clean_path = os.path.join(data, "clean.csv")
balanced_path = os.path.join(data, "balanced.csv")

clean_dataset = pd.read_csv(filepath_or_buffer=clean_path)
balanced_dataset = pd.read_csv(filepath_or_buffer=balanced_path)

## Modeling: Clean Dataset (Imbalance)

#### Split dataset to inputs (x), target (y) for train, test and validation

In [4]:
X = clean_dataset.drop("Price", axis=1)
y = clean_dataset["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

#### Compute metrics

1. Cross Entropy Loss
2. Accuracy
3. Precision
4. Recall
5. F1 Score

In [5]:
def compute_eval_metrics(actual, prediction,  probabilities) -> tuple: #TODO
    """
    TODO probabilities
    :param actual:
    :param prediction:
    :param probabilities:
    :return:
    """
    loss = log_loss(actual, probabilities)
    accuracy = accuracy_score(actual, prediction)
    precision = precision_score(actual, prediction, average="macro")
    recall = recall_score(actual, prediction, average="macro")
    f1 = f1_score(actual, prediction, average="macro")

    return round(float(loss), 2), round(float(accuracy), 2), round(float(precision), 2), round(float(recall), 2), round(float(f1), 2)

#### Tuning the hyperparameters utility

In [6]:
def objective(trial, model_name, model, X_train, y_train, X_validation, y_validation): #TODO
    """
    TODO
    :param trial:
    :param model_name:
    :param model:
    :param X_train:
    :param y_train:
    :param X_validation:
    :param y_validation:
    :return:
    """
    if model_name == "SoftMax Regression":
        class_weight = trial.suggest_categorical("class_weight", ["balanced"])
        multi_class = trial.suggest_categorical("multi_class", ["multinomial"])
        solver = trial.suggest_categorical("solver", ["lbfgs"])
        c = trial.suggest_loguniform("C", 0.1, 1)
        max_iter = trial.suggest_int("max_iter", 5000, 10000)

        model.set_params(class_weight=class_weight, multi_class=multi_class, solver=solver, C=c, max_iter=max_iter)

    elif model_name == "SVC":
        decision_function_shape = trial.suggest_categorical("decision_function_shape", ["ovo"])
        class_weight = trial.suggest_categorical("class_weight", ["balanced"])
        kernel = trial.suggest_categorical("kernel", ["linear", "rbf", "poly"])
        gamma = trial.suggest_float("gamma", 0.1, 1.0)
        c = trial.suggest_loguniform("C", 0.1, 1)
        max_iter = trial.suggest_int("max_iter", 5000, 10000)

        model.set_params(decision_function_shape=decision_function_shape, class_weight=class_weight, kernel=kernel, gamma=gamma, C=c, max_iter=max_iter)


    if model_name == "Random Forest Classifier":
        class_weight = trial.suggest_categorical("class_weight", ["balanced"])
        n_estimators = trial.suggest_int("n_estimators", 100, 500)
        max_depth = trial.suggest_int("max_depth", 5, 50)
        criterion = trial.suggest_categorical("criterion", ["log_loss"])

        model.set_params(class_weight=class_weight, n_estimators=n_estimators, max_depth=max_depth, criterion=criterion)


    model.fit(X_train, y_train)

    predictions = model.predict(X_validation)
    probabilities = model.predict_proba(X_validation)

    loss, accuracy, precision, recall, f1 = compute_eval_metrics(y_validation, predictions, probabilities)

    return f1

#### Initialize models
1. SoftMax Regression
2. Support Vector Classifier (SVC)
3. Random Forest Classifier

In [7]:
models = [("SoftMax Regression", LogisticRegression(random_state=42), (X_train, y_train), (X_validation, y_validation), (X_test, y_test)),
          ("SVC", SVC(random_state=42, probability=True), (X_train, y_train), (X_validation, y_validation), (X_test, y_test)),
          ("Random Forest Classifier", RandomForestClassifier(random_state=42), (X_train, y_train), (X_validation, y_validation), (X_test, y_test))]

#### Training and evaluating the model performance, and logging the results and configurations

In [8]:
for model_name, model, train, validation, test in models:
    X_train, y_train = train
    X_validation, y_validation = validation
    X_test, y_test = test

    start = time()

    study = optuna.create_study(direction="maximize", study_name=model_name)
    study.optimize(lambda trial: objective(trial, model_name, model, X_train, y_train, X_validation, y_validation), n_trials=50)

    best_params = study.best_params
    model.set_params(**best_params)

    model.fit(X_train, y_train)

    end = time()

    predictions_validation = model.predict(X_validation)
    probabilities_validation = model.predict_proba(X_validation)

    loss_validation, accuracy_validation, precision_validation, recall_validation, f1_validation = compute_eval_metrics(y_validation,predictions_validation,probabilities_validation)

    predictions_test = model.predict(X_test)
    probabilities_test = model.predict_proba(X_test)

    loss_test, accuracy_test, precision_test, recall_test, f1_test = compute_eval_metrics(y_test, predictions_test, probabilities_test)

    print(f"Model: {model_name} >>"
          f" Loss (Validation): {loss_validation},"
          f" Accuracy (Validation): {accuracy_validation},"
          f" Precision (Validation): {precision_validation},"
          f" Recall (Validation): {recall_validation},"
          f" F1 Score (Validation): {f1_validation}")

    print(f"Model: {model_name} >>"
          f" Loss (Test): {loss_test},"
          f" Accuracy (Test): {accuracy_test},"
          f" Precision (Test): {precision_test},"
          f" Recall (Test): {recall_test},"
          f" F1 Score (Test): {f1_test}")

    path = os.path.join(experiments, model_name)
    mlflow.set_tracking_uri(path)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)

        mlflow.log_metrics({"Validation Loss": loss_validation,
                            "Validation Accuracy": accuracy_validation,
                            "Validation Precision": precision_validation,
                            "Validation Recall": recall_validation,
                            "Validation F1": f1_validation,
                            "Test Loss": loss_test,
                            "Test Accuracy": accuracy_test,
                            "Test Precision": precision_test,
                            "Test Recall": recall_test,
                            "Test F1": f1_test,
                            "Training Set": X_train.shape[0],
                            "Validation Set": X_validation.shape[0],
                            "Test Set": X_test.shape[0]})

        mlflow.set_tags({"Dataset": "Clean",
                         "Size": clean_dataset.shape[0],
                         "Training Period": f"{round((end - start) / 3600, 2)} Hours"})

        mlflow.sklearn.log_model(model, "model", input_example=X_test.iloc[[0]])

Model: SoftMax Regression >> Loss (Validation): 1.56, Accuracy (Validation): 0.4, Precision (Validation): 0.16, Recall (Validation): 0.18, F1 Score (Validation): 0.16
Model: SoftMax Regression >> Loss (Test): 1.62, Accuracy (Test): 0.4, Precision (Test): 0.17, Recall (Test): 0.23, F1 Score (Test): 0.17
Model: SVC >> Loss (Validation): 1.18, Accuracy (Validation): 0.39, Precision (Validation): 0.17, Recall (Validation): 0.22, F1 Score (Validation): 0.18
Model: SVC >> Loss (Test): 1.18, Accuracy (Test): 0.42, Precision (Test): 0.16, Recall (Test): 0.18, F1 Score (Test): 0.16
Model: Random Forest Classifier >> Loss (Validation): 1.28, Accuracy (Validation): 0.59, Precision (Validation): 0.2, Recall (Validation): 0.23, F1 Score (Validation): 0.21
Model: Random Forest Classifier >> Loss (Test): 1.28, Accuracy (Test): 0.56, Precision (Test): 0.22, Recall (Test): 0.25, F1 Score (Test): 0.22


In [9]:
X = balanced_dataset.drop("Price", axis=1)
y = balanced_dataset["Price"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_validation, y_train, y_validation = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)

In [10]:
models = [("SoftMax Regression", LogisticRegression(random_state=42), (X_train, y_train), (X_validation, y_validation), (X_test, y_test)),
          ("SVC", SVC(random_state=42, probability=True), (X_train, y_train), (X_validation, y_validation), (X_test, y_test)),
          ("Random Forest Classifier", RandomForestClassifier(random_state=42), (X_train, y_train), (X_validation, y_validation), (X_test, y_test))]

In [11]:
for model_name, model, train, validation, test in models:
    X_train, y_train = train
    X_validation, y_validation = validation
    X_test, y_test = test

    start = time()

    study = optuna.create_study(direction="maximize", study_name=model_name)
    study.optimize(lambda trial: objective(trial, model_name, model, X_train, y_train, X_validation, y_validation), n_trials=50)

    best_params = study.best_params
    model.set_params(**best_params)

    model.fit(X_train, y_train)

    end = time()

    predictions_validation = model.predict(X_validation)
    probabilities_validation = model.predict_proba(X_validation)

    loss_validation, accuracy_validation, precision_validation, recall_validation, f1_validation = compute_eval_metrics(y_validation,predictions_validation,probabilities_validation)

    predictions_test = model.predict(X_test)
    probabilities_test = model.predict_proba(X_test)

    loss_test, accuracy_test, precision_test, recall_test, f1_test = compute_eval_metrics(y_test, predictions_test, probabilities_test)

    print(f"Model: {model_name} >>"
          f" Loss (Validation): {loss_validation},"
          f" Accuracy (Validation): {accuracy_validation},"
          f" Precision (Validation): {precision_validation},"
          f" Recall (Validation): {recall_validation},"
          f" F1 Score (Validation): {f1_validation}")

    print(f"Model: {model_name} >>"
          f" Loss (Test): {loss_test},"
          f" Accuracy (Test): {accuracy_test},"
          f" Precision (Test): {precision_test},"
          f" Recall (Test): {recall_test},"
          f" F1 Score (Test): {f1_test}")

    path = os.path.join(experiments, model_name)
    mlflow.set_tracking_uri(path)

    with mlflow.start_run(run_name=model_name):
        mlflow.log_params(best_params)

        mlflow.log_metrics({"Validation Loss": loss_validation,
                            "Validation Accuracy": accuracy_validation,
                            "Validation Precision": precision_validation,
                            "Validation Recall": recall_validation,
                            "Validation F1": f1_validation,
                            "Test Loss": loss_test,
                            "Test Accuracy": accuracy_test,
                            "Test Precision": precision_test,
                            "Test Recall": recall_test,
                            "Test F1": f1_test,
                            "Training Set": X_train.shape[0],
                            "Validation Set": X_validation.shape[0],
                            "Test Set": X_test.shape[0]})

        mlflow.set_tags({"Dataset": "Balanced",
                         "Size": balanced_dataset.shape[0],
                         "Training Period": f"{round((end - start) / 3600, 2)} Hours"})

        mlflow.sklearn.log_model(model, "model", input_example=X_test.iloc[[0]])

Model: SoftMax Regression >> Loss (Validation): 0.99, Accuracy (Validation): 0.67, Precision (Validation): 0.65, Recall (Validation): 0.67, F1 Score (Validation): 0.65
Model: SoftMax Regression >> Loss (Test): 0.95, Accuracy (Test): 0.68, Precision (Test): 0.66, Recall (Test): 0.68, F1 Score (Test): 0.67
Model: SVC >> Loss (Validation): 0.05, Accuracy (Validation): 0.98, Precision (Validation): 0.98, Recall (Validation): 0.98, F1 Score (Validation): 0.98
Model: SVC >> Loss (Test): 0.05, Accuracy (Test): 0.98, Precision (Test): 0.98, Recall (Test): 0.98, F1 Score (Test): 0.98
Model: Random Forest Classifier >> Loss (Validation): 0.23, Accuracy (Validation): 0.98, Precision (Validation): 0.98, Recall (Validation): 0.98, F1 Score (Validation): 0.98
Model: Random Forest Classifier >> Loss (Test): 0.23, Accuracy (Test): 0.97, Precision (Test): 0.97, Recall (Test): 0.97, F1 Score (Test): 0.97
